# CodeAct REPL and Pass by Reference

In this notebook we'll use NOOA to build a `BookshopAgent` on top of the full Project Gutenberg catalog — tens of thousands of rows of real titles, authors, and languages. To recommend a book, the agent has to **explore the catalog** — but the catalog is far too big to just drop into the prompt and let the model read straight through.

The idea we'll land on is old-fashioned: **pass by reference**.

The model doesn't need to see every row of the catalog. It needs *verbs* — `search_titles`, `by_topic`, `by_author`. The DataFrame stays in Python memory, and the model reaches it through methods you wrote. That's the pattern that makes CodeAct scale.


## Prerequisites

Run the install cell below before the setup cell.

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, set your key in Colab's Secrets (🔑 in the left sidebar) as `API_KEY`. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


In [ ]:
!pip install nooa


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. For hosted providers, set your key in Colab's Secrets (🔑 in the sidebar) as `API_KEY`. Local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# In Colab: click the 🔑 icon in the left sidebar, add a secret called API_KEY,
# and enable "Notebook access". Then:
from google.colab import userdata
api_key = userdata.get("API_KEY")

# Locally: `export API_KEY=...` in your shell, then uncomment:
# import os
# api_key = os.environ["API_KEY"]

# model = get_llm_client("claude-haiku-4-5", api_key=api_key)                          # Anthropic
model = get_llm_client("gpt-5.5", api_key=api_key)                                     # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")   # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1") # vLLM (local, no key)


## Loading the Catalog

We'll grab the Project Gutenberg catalog: title, author, language, subjects, bookshelves — a few tens of thousands of public-domain books. The upstream CSV uses `Title`, `Authors`, `Subjects`; we rename to snake_case as we load.

In [2]:
import pandas as pd
from pydantic import BaseModel

# `doc` renders the same view of an agent that the LLM sees.
from nooa import Agent, print_prompt
from nooa.agentdoc import doc

The compressed CSV lives on Project Gutenberg's servers. We fetch it and keep just the columns we care about.

In [3]:
CATALOG_URL = "https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv.gz"

raw_catalog = pd.read_csv(CATALOG_URL, compression="gzip")

catalog = raw_catalog.rename(
    columns={
        "Text#": "id", "Title": "title", "Authors": "author",
        "Language": "language", "Subjects": "topics", "Bookshelves": "bookshelves",
    }
)[["id", "title", "author", "language", "topics", "bookshelves"]]

How big is this thing? The token estimate at the bottom is the number to remember — it's what we'd be paying, per call, if we shoved the whole catalog into the prompt.

In [ ]:
# ~4 chars per token is the usual back-of-envelope figure.
csv_chars = len(catalog.to_csv(index=False))

print(f"Catalog source: {CATALOG_URL}")
print(f"Catalog rows: {len(catalog):,}")
print(f"In-memory footprint: {catalog.memory_usage(deep=True).sum():,} bytes")
print(f"Serialized as CSV:   ~{csv_chars // 4:,} tokens, very roughly")
catalog.head(3)

Catalog source: https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv.gz
Catalog rows: 78,999
In-memory footprint: 38,786,776 bytes
Serialized as CSV:   ~4,873,936 tokens, very roughly

,id,title,author,language,topics,bookshelves
0,1,The Declaration of Independence of the United ...,"Jefferson, Thomas, 1743-1826",en,"United States -- History -- Revolution, 1775-1...",Politics; American Revolutionary War; United S...
1,2,The United States Bill of Rights\r\nThe Ten Or...,United States,en,Civil rights -- United States -- Sources; Unit...,Politics; American Revolutionary War; United S...
2,3,John F. Kennedy's Inaugural Address,"Kennedy, John F. (John Fitzgerald), 1917-1963",en,United States -- Foreign relations -- 1961-196...,"Category: Essays, Letters & Speeches; Category..."


## Designing the Agent's Surface

The catalog is a `pd.DataFrame`, and CodeAct hands the model a Python REPL — so in principle we could attach it to `self` and let the model use pandas directly. The trouble isn't that the DataFrame's *rows* would flood the prompt. It's that once the model has a live `DataFrame` in scope, it's tempted to reach for pandas directly: filter, join, slice, compute. That's a lot of surface, and the model doesn't need most of it to answer a customer.

The design that scales is the ordinary object-oriented one: **the model works with the DataFrame by reference, through a small set of verbs you wrote.** No serialization, no copy — the object stays put in Python, and the methods reach it.

- `self._catalog` — the DataFrame lives here, on the agent, as a live Python object. The model doesn't get a copy; its methods operate on this one reference.
- A handful of deterministic verbs — `search_titles`, `by_topic`, `in_language`, `by_author` — that filter the DataFrame and hand back small result sets.
- A CodeAct method (`recommend`) that composes them.

The structured output the recommender returns:


In [5]:
class Recommendation(BaseModel):
    """The final recommendation returned to the caller."""
    title: str
    author: str
    language: str
    why_this_book: str

Let's start with just the state and one search helper. `self._catalog` holds the DataFrame — one object, in Python memory. Every helper we're about to write reaches into that same reference.

In [ ]:
class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and recommend public-domain books."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        # The DataFrame lives here as a live Python object. The helpers below
        # reach into this one reference — no copy, no serialization.
        self._catalog = catalog

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
        q = query.lower()
        mask = self._catalog["title"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

Now the rest of the helpers — three more verbs, one per useful axis of the catalog. Notice these are **plain, deterministic Python methods**: no LLM, no I/O beyond pandas — same inputs, same outputs, every time. The model doesn't run them; it just calls them. The only non-deterministic method on this agent will be `recommend`, which is the one that actually consults the LLM.


In [7]:
class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and recommend public-domain books."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        self._catalog = catalog

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
        q = query.lower()
        mask = self._catalog["title"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention topic."""
        q = topic.lower()
        # Combined search across both subject columns.
        text = self._catalog["topics"].fillna("") + " " + self._catalog["bookshelves"].fillna("")
        mask = text.str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def in_language(self, language_code: str, n: int = 10) -> list[dict]:
        """Return up to n books in the given ISO 639-1 language code (en, fr, de, ...)."""
        mask = self._catalog["language"].str.lower() == language_code.lower()
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_author(self, author: str, n: int = 10) -> list[dict]:
        """Return up to n books whose author field contains author."""
        q = author.lower()
        mask = self._catalog["author"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

And the CodeAct method — `recommend`. Its docstring points the model at the helpers and away from any attempt to see the whole catalog.


In [8]:
class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and recommend public-domain books."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        self._catalog = catalog

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
        q = query.lower()
        mask = self._catalog["title"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention topic."""
        q = topic.lower()
        text = self._catalog["topics"].fillna("") + " " + self._catalog["bookshelves"].fillna("")
        mask = text.str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def in_language(self, language_code: str, n: int = 10) -> list[dict]:
        """Return up to n books in the given ISO 639-1 language code (en, fr, de, ...)."""
        mask = self._catalog["language"].str.lower() == language_code.lower()
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_author(self, author: str, n: int = 10) -> list[dict]:
        """Return up to n books whose author field contains author."""
        q = author.lower()
        mask = self._catalog["author"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    async def recommend(self, customer_wants: str) -> Recommendation:
        """Find one public-domain book for this customer.

        Use self.search_titles, self.by_topic, self.in_language, and self.by_author
        to explore the catalog. Do not ask for or print the whole catalog. Return
        one recommendation with a short, concrete reason.
        """
        ...

## What the Model Sees

`doc(agent)` renders the view of the agent that the framework hands to the LLM.

In [9]:
agent = BookshopAgent(catalog)

print(doc(agent))

class BookshopAgent:
    """You run a used bookshop and recommend public-domain books."""

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention topic."""
    def in_language(self, language_code: str, n: int = 10) -> list[dict]:
        """Return up to n books in the given ISO 639-1 language code (en, fr, de, ...)."""
    def by_author(self, author: str, n: int = 10) -> list[dict]:
        """Return up to n books whose author field contains author."""
    async def recommend(self, customer_wants: str) -> Recommendation:
        """
        Find one public-domain book for this customer.
        
        Use self.search_titles, self.by_topic, self.in_language, and self.by_author
        to explore the catalog. Do not ask for or print the whole catalog. Return
        one recommendation with a short, concrete reason.
        """
## Referenced Types
class Recommendation(BaseModel):
    """The final recommendation returned to the caller."""

    title: str
    author: str
    language: str
    why_this_book: str

The four search helpers show up, along with `recommend`. The DataFrame itself doesn't — it's still there in Python memory, but the model reaches it only through the verbs. That's the pass-by-reference pattern: one live object on the Python side, a small API on the model side.

### 🥷 Under the hood

Let's sneak up on the model again. `print_prompt` shows the exact prompt for a `recommend` call — and we'll print it alongside the token count the catalog *would* have added if it had been part of the prompt.

In [10]:
# The counterfactual: what serializing the catalog would cost, per call.
would_be_chars = len(repr(catalog.to_dict(orient="records")))
print(f"If serialized directly, the catalog would be roughly {would_be_chars // 4:,} tokens.\n")

# The actual prompt the model receives.
await print_prompt(agent.recommend, customer_wants="a short English mystery, nothing too heavy")

If serialized directly, the catalog would be roughly 6,391,033 tokens.

=== SYSTEM PROMPT  [BookshopAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You run a used bookshop and recommend public-domain books.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — never construct large arrays by hand.

For language tasks (classification, extraction, interpretation), use LLM reasoning — answer directly via `return_result`, or delegate to a `@strategy(PredictStrategy())` standalone function (see below). Don't keyword-match or regex.

## Returning computed results

After computing in code, call `return_result(variable)` **from within** `execute_python()`. This passes the variable directly. Do NOT re-type computed values in a separate `return_result` tool call.

## Helpers

Define helpers at the top of the cell and call them by name. Existing methods on `self` are usable via `await self.method(...)`. Helpers persist as REPL locals across cells in this session.

```python
def normalize(x):
    return x.strip().lower()

cleaned = [normalize(v) for v in values]
```

## Fan-out generation

For per-item LLM work over a list, decorate a standalone async function with `@strategy(PredictStrategy())` and an ellipsis body. `asyncio.gather` runs the calls in parallel.

```python
@strategy(PredictStrategy())
async def detect_language(message: str) -> str:
    """Return the ISO 639-1 language code for {message} (e.g. 'en', 'fr', 'de', 'ja')."""
    ...

codes = await asyncio.gather(*(detect_language(m) for m in messages))
return_result(codes)
```

For iterative sub-tasks that need code execution, use `@strategy(CodeActStrategy())`. The sub-task must be strictly simpler than the current call to avoid infinite recursion.

## Restrictions (will throw)

- `eval`, `exec`, `compile`, `__import__`, `input`, `breakpoint`
- `globals`, `locals`, `vars`, `asyncio.run`, `loop.run_until_complete`
- Attaching callables to the agent: `self.foo = fn`, `setattr(self, 'foo', fn)`, `type(self).foo = fn`
</strategy_prompt>

<execution_context>
## Execution Context

These names are already in scope inside `execute_python()` (state persists across cells) — call them, don't re-import or re-define. Use `doc(name)` to inspect any type or function in detail.

```python
import pandas as pd
from nooa import Agent, print_prompt
from nooa.agentdoc import doc
from nooa.unifiedllm import get_llm_client
from pandas import DataFrame
from pydantic import BaseModel

class BookshopAgent: ...
class Recommendation: ...
```
Also in scope: exit, get_ipython, open, quit.
Always available without import: `self`, `print()`, `pprint()`, `doc()`, `return_result()`, plus stdlib `asyncio` and `typing`.
</execution_context>

<self expr="doc(type(self))">
class BookshopAgent:
    """You run a used bookshop and recommend public-domain books."""

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention topic."""
    def in_language(self, language_code: str, n: int = 10) -> list[dic

The prompt is method signatures, docstrings, and the argument. The catalog is gone from it — replaced by the four verbs that can reach into it. Every call. Not just the first one.

> 📝 **Takeaway:** the DataFrame is passed by reference, not by value. The model works with a live Python object through methods you designed on purpose, instead of receiving a serialized snapshot each turn.

## The Helpers Are Just Python

The helpers are ordinary methods — no LLM in the loop. You can call them, test them, compose them.

In [11]:
# A direct method call. No prompt, no LLM.
hits = agent.by_topic("mystery", n=20)

# Compose with plain Python.
short_titles = [book for book in hits if len(str(book["title"])) < 45]

print(f"Mystery hits: {len(hits)}")
print(f"Short title candidates: {len(short_titles)}")
short_titles[:3]

Mystery hits: 20
Short title candidates: 19

[{'id': 42,
  'title': 'The strange case of Dr. Jekyll and Mr. Hyde',
  'author': 'Stevenson, Robert Louis, 1850-1894',
  'language': 'en',
  'topics': 'Science fiction; Horror tales; London (England) -- Fiction; Physicians -- Fiction; Psychological fiction; Self-experimentation in medicine -- Fiction; Multiple personality -- Fiction',
  'bookshelves': 'Precursors of Science Fiction; Horror; Gothic Fiction; Movie Books; Category: Crime, Thrillers and Mystery; Category: Novels; Category: Classics of Literature; Category: British Literature'},
 {'id': 43,
  'title': 'The strange case of Dr. Jekyll and Mr. Hyde',
  'author': 'Stevenson, Robert Louis, 1850-1894',
  'language': 'en',
  'topics': 'Science fiction; Horror tales; London (England) -- Fiction; Physicians -- Fiction; Psychological fiction; Self-experimentation in medicine -- Fiction; Multiple personality -- Fiction',
  'bookshelves': 'Precursors of Science Fiction; Horror; Movie Books; Category: Crime, Thrillers and Mystery; Category: Novels; Category: Classics of Literature; Category: British Literature'},
 {'id': 79,
  'title': 'Terminal Compromise',
  'author': 'Schwartau, Winn, 1952-',
  'language': 'en',
  'topics': 'Computer security -- Fiction; Didactic fiction; Privacy, Right of -- Fiction; Records -- Access control -- Fiction',
  'bookshelves': 'Category: Crime, Thrillers and Mystery; Category: Novels; Category: American Literature'}]

## Running `recommend`

The LLM will use the same helpers, only from inside a CodeAct REPL, chained together with judgment at each step.

In [12]:
rec = await agent.recommend("a short English mystery, nothing too heavy")

print(f"Recommended: {rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")

Recommended: The Return of Sherlock Holmes by Doyle, Arthur Conan, 1859-1930 (en)
Why: A collection of brisk Sherlock Holmes detective stories in English, so it fits a short mystery mood without the weight of a long, heavy novel.

If the trace viewer isn't already running, start it in a terminal:

```bash
nooa start-dev
```

Re-run the recommendation and inspect the CodeAct trace. Look at the boundary: generated Python calls `self.by_topic(...)` and `self.search_titles(...)`, but the catalog itself is nowhere in the prompt. Only the small helper results come back into the model's context.

## Aside: CodeAct Is a REPL, Not Stateless Tool Calling

Worth naming the mechanic behind all of this. In a stateless tool-call loop, every tool call is a message and every result is a message; variables don't persist between calls.

CodeAct is a persistent Python session. Variables assigned in one generated cell are still there in the next:

```python
# First cell:
hits = self.by_topic("mystery", n=50)
short_titles = [b for b in hits if len(b["title"]) < 45]
```

```python
# Later cell — short_titles is still bound:
best = short_titles[0]
return_result({"title": best["title"], ...})
```

That persistence is what makes the small-API pattern pay off. The model builds up small working sets in Python; the large object stays put on `self._catalog`.

## Next: Making Visibility Explicit

We kept the DataFrame off the agent's model-facing surface by prefixing it with `_`, without really explaining the rules behind that. Notebook 4 is dedicated to those rules — `@hidden`, `Annotated[T, hidden]`, `@spec(hidden=False)`, the `with hidden:` context manager, and the underscore convention. Different levers for different situations, same underlying idea: you decide what the model sees.

## Recap

- **Pass by reference.** The DataFrame lives once, on the agent, in Python memory. The model doesn't get a copy — it operates on the live object through methods.
- **The model reaches state through a small API.** A handful of deterministic verbs beats handing the container over.
- **Helpers are just Python.** Test them, compose them, refactor them — no prompt engineering required.
- **`doc(self)` is the model's view of the object.** If it isn't in there, the model can't see it. (Notebook 4 covers the visibility rules that decide what lands in there.)
- **CodeAct is a persistent REPL.** Variables survive across generated cells, so the model can build up small working sets while the large object stays put.

## Exercises

1. **A new helper.** Add `recent_books(n: int = 10) -> list[dict]` using the Gutenberg id as a rough proxy for recency, then ask `recommend` for a recent-feeling book.
2. **A helper that shapes the model's judgment.** Add `count_by_author(author: str) -> int` and update `recommend`'s docstring to prefer prolific authors when the request is vague.
3. **Confirm the visibility rule.** Hide one of the search helpers explicitly (`from nooa import hidden; @hidden`, a peek at notebook 4). Print `doc(agent)` and check that it's gone.
4. **Keep tool results small.** Add a `titles_only` variant of one search helper that returns only `title`, `author`, `language`. Smaller results, cheaper CodeAct turns.